# Auto-Ingestion: From Database to Semantic Models

TL;DR: SLayer can introspect a database schema and automatically generate models with dimensions, measures, and join relationships matching the schema.  

This notebook shows what happens under the hood.

We use the **Jaffle Shop** dataset — a synthetic e-commerce schema with 7 tables and foreign key relationships between them.

**Prerequisites:** `pip install motley-slayer[examples]` (jafgen is installed by the cell below)

See also: [Auto-Ingestion docs](../../concepts/ingestion.md) | [Models docs](../../concepts/models.md)



In [1]:
# Install jafgen (Jaffle Shop data generator) from a specific commit
# The released PyPI version has a bug; this pinned commit is the fix.
# This is only needed for running the tutorials — not a SLayer dependency.
!pip install -q git+https://github.com/rossbowen/jaffle-shop-generator.git@09557a1118b000071f8171aa97d54d5029bf0f0b


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import sys

import duckdb
import sqlalchemy as sa

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop, DB_PATH

engine, storage, models = ensure_jaffle_shop()

## The Jaffle Shop Schema

The database has 7 tables connected by foreign keys:

```
customers  <──  orders  ──>  stores
                  │
            order_items  ──>  products  <──  supplies

customers  <──  tweets
```

Let's peek at the data:

In [3]:
conn = duckdb.connect(DB_PATH, read_only=True)
for table in ["customers", "stores", "products", "orders", "order_items", "supplies", "tweets"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:>15}: {count:>8,} rows")
conn.close()

      customers:    1,902 rows
         stores:        6 rows
       products:       10 rows
         orders:  206,136 rows
    order_items:  326,064 rows
       supplies:       65 rows
         tweets:   99,231 rows


## Step 1: FK Graph Discovery

The first thing auto-ingestion does is introspect foreign key constraints and build a **directed dependency graph**. Each edge means "this table has a FK pointing to that table."

The graph must be acyclic — if cycles are found, SLayer raises a `RollupGraphError`.

For each table, SLayer then computes the **transitive closure**: all tables reachable by following FK chains. This determines which joined dimensions will be available.

In [4]:
from slayer.core.models import DatasourceConfig
from slayer.engine.ingestion import _build_fk_graph, _compute_transitive_closure

ds = DatasourceConfig(name="jaffle_shop", type="duckdb", database=DB_PATH)
sa_engine = sa.create_engine(ds.resolve_env_vars().get_connection_string())
inspector = sa.inspect(sa_engine)
table_names = inspector.get_table_names()

fk_graph = _build_fk_graph(inspector=inspector, table_names=table_names, schema=None)

print("FK Graph (table -> tables it references):")
for table in sorted(fk_graph):
    refs = sorted(fk_graph[table])
    print(f"  {table} -> {refs}")

print()
print("Tables with no FK references (leaf tables):")
for t in sorted(table_names):
    if t not in fk_graph:
        print(f"  {t}")

FK Graph (table -> tables it references):
  order_items -> ['orders', 'products']
  orders -> ['customers', 'stores']
  supplies -> ['products']
  tweets -> ['customers']

Tables with no FK references (leaf tables):
  customers
  products
  stores


In [5]:
# Transitive closure: what tables are reachable from order_items?
for table in ["orders", "order_items"]:
    reachable = _compute_transitive_closure(fk_graph, table)
    print(f"{table} can reach: {sorted(reachable)}")

sa_engine.dispose()

orders can reach: ['customers', 'stores']
order_items can reach: ['customers', 'orders', 'products', 'stores']


Notice that `order_items` transitively reaches `customers` and `stores` through `orders`, even though it has no direct FK to them.

## Step 2: Join Generation

SLayer converts FK relationships into `ModelJoin` objects via BFS. Each join specifies:
- **target_model**: the table being joined
- **join_pairs**: `[[source_column, target_column]]` — the columns to join on

For multi-hop joins (e.g., `order_items -> orders -> customers`), the source column is path-qualified: `"orders.customer_id"` means "the `customer_id` column in the already-joined `orders` table."

In [6]:
orders_model = next(m for m in models if m.name == "orders")
oi_model = next(m for m in models if m.name == "order_items")

print("=== orders model joins (2 direct FKs) ===")
for j in orders_model.joins:
    print(f"  -> {j.target_model}  ON {j.join_pairs}")

print()
print("=== order_items model joins (2 direct + 2 multi-hop) ===")
for j in oi_model.joins:
    src = j.join_pairs[0][0]
    tgt = j.join_pairs[0][1]
    multi_hop = " (multi-hop)" if "." in src else ""
    print(f"  -> {j.target_model}  ON {src} = {tgt}{multi_hop}")

=== orders model joins (2 direct FKs) ===
  -> customers  ON [['customer_id', 'id']]
  -> stores  ON [['store_id', 'id']]

=== order_items model joins (2 direct + 2 multi-hop) ===
  -> orders  ON order_id = id
  -> products  ON sku = sku
  -> customers  ON orders.customer_id = id (multi-hop)
  -> stores  ON orders.store_id = id (multi-hop)


## Step 3: Dimension & Measure Generation

For each table, SLayer generates:
- **Dimensions** for every column. See the [joins](../05_joins/joins.ipynb) on how to refer to joined dimensions and measures. 
- **Measures**: `count` always; one measure per numeric non-ID column (with `sql` set to the column name); non-numeric non-ID columns also get a measure

In [7]:
print("=== orders model dimensions ===")
print(f"{'Name':<25} {'SQL':<25} {'Type':<8}")
print("-" * 60)
for dim in orders_model.dimensions:
    pk = " [PK]" if dim.primary_key else ""
    print(f"{dim.name:<25} {dim.sql:<25} {dim.type:<8}{pk}")

=== orders model dimensions ===
Name                      SQL                       Type    
------------------------------------------------------------
id                        id                        string   [PK]
customer_id               customer_id               string  
ordered_at                ordered_at                date    
store_id                  store_id                  string  


In [8]:
print("=== orders model measures ===")
print(f"{'Name':<30} {'SQL':<20}")
print("-" * 52)
for m in orders_model.measures:
    sql_str = m.sql or "(COUNT(*))"
    print(f"{m.name:<30} {sql_str:<20}")

=== orders model measures ===
Name                           SQL                 
----------------------------------------------------
subtotal                       subtotal            
tax_paid                       tax_paid            
order_total                    order_total         
ordered_at                     ordered_at          


Joined dimensions aren't stored on the model — they're resolved at query time.

Use dot syntax to walk the join graph: 1-hop or multi-hop.

In [9]:
# 1-hop: order_items -> products
result = engine.execute(
    query={
        "source_model": "order_items",
        "fields": ["*:count", "quantity:sum"],
        "dimensions": ["products.name"],
        "order": [{"column": {"name": "quantity_sum"}, "direction": "desc"}],
        "limit": 5,
    }
)

print("=== 1-hop: order_items -> products ===")
for row in result.data:
    print(
        f"  {row['order_items.products.name']}: {row['order_items._count']} items, {row['order_items.quantity_sum']} qty"
    )

# Multi-hop: order_items -> orders -> customers
result = engine.execute(
    query={
        "source_model": "order_items",
        "fields": ["*:count", "quantity:sum"],
        "dimensions": ["orders.customers.name"],
        "order": [{"column": {"name": "quantity_sum"}, "direction": "desc"}],
        "limit": 5,
    }
)

print("\n=== Multi-hop: order_items -> orders -> customers ===")
for row in result.data:
    print(
        f"  {row['order_items.orders.customers.name']}: {row['order_items._count']} items, {row['order_items.quantity_sum']} qty"
    )

=== 1-hop: order_items -> products ===
  adele-ade: 46265 items, 48091 qty
  for richer or pourover : 46201 items, 48052 qty
  chai and mighty: 46215 items, 47996 qty
  tangaroo: 46064 items, 47943 qty
  vanilla ice: 45813 items, 47603 qty

=== Multi-hop: order_items -> orders -> customers ===
  David Evans: 943 items, 962 qty
  Ryan Duarte: 852 items, 852 qty
  Andrea Thompson: 835 items, 846 qty
  Robert Good: 839 items, 839 qty
  Theresa Fox: 822 items, 822 qty


## Summary

Auto-ingestion discovers the database structure and generates a complete semantic layer:

1. **FK graph** identifies table relationships and validates they're acyclic
2. **Transitive closure** determines which tables should be joined (including multi-hop)
3. **Join metadata** (`ModelJoin` objects) captures how to join tables at query time
4. **Dimensions and measures** are generated with proper naming conventions

No join-related SQL is baked into the models — JOINs are constructed dynamically at query time from the join metadata.

**Next:** See the [Example Queries](../example_queries/example_queries.ipynb) notebook to query this data, or the [Joins](../05_joins/joins.ipynb) notebook for a deep dive into join mechanics.